In [3]:
import sys
sys.path.append('/var/www/python/Prod/nighthawk/')

import pandas as pd
from pathlib import Path
from nighthawk.data import Constraint

CSV_PATH  = Path('/var/www/python/Qingcheng/QCTest/Manual_bidding/update_sheet/Daily Bidding - daily_constraint.csv')
MARKET    = 'SPP'
THRESHOLD = -900
META_COLS = ['location', 'physical_condition', 'outage_name',
             'comment on this constraint', 'start_date', 'end_date',
             'wind', 'reserve_zone']
COL_ORDER = ['bid_date', 'monitored', 'DA_mvalue', 'RT_mvalue',
             'location', 'physical_condition', 'outage_name',
              'start_date', 'end_date',
             'wind', 'reserve_zone', "today's wind", 'opportunity']


def _fetch_mvalues(dt_str: str) -> pd.DataFrame:
    opex  = MARKET
    mv_rt = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
        start_dt=dt_str, end_dt=dt_str, type='RT', granularity='daily')
    mv_da = Constraint(oops_constraint_num_df=None, market=opex).get_mvalues(
        start_dt=dt_str, end_dt=dt_str, type='DA', granularity='daily')

    all_cons = pd.DataFrame({'oops_constraint_num':
        pd.concat([mv_rt['oops_constraint_num'], mv_da['oops_constraint_num']]).unique()})

    det_rt = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='RT')
    det_da = Constraint(oops_constraint_num_df=all_cons, market=opex).get_constraint_details(da_or_rt='DA')
    details = (pd.concat([det_rt, det_da])
               .drop_duplicates('oops_constraint_num')
               [['oops_constraint_num', 'monitored_clean']]
               .rename(columns={'monitored_clean': 'monitored'}))

    da_sum = (mv_da.merge(details, on='oops_constraint_num', how='left')
              .groupby('monitored')['mvalue'].sum().rename('DA_mvalue').reset_index())
    rt_sum = (mv_rt.merge(details, on='oops_constraint_num', how='left')
              .groupby('monitored')['mvalue'].sum().rename('RT_mvalue').reset_index())

    merged = pd.merge(da_sum, rt_sum, on='monitored', how='outer').fillna(0)
    merged['monitored'] = merged['monitored'].str.strip()
    merged['DA_mvalue'] = merged['DA_mvalue'].round(0).astype(int)
    merged['RT_mvalue'] = merged['RT_mvalue'].round(0).astype(int)
    return merged


def _lookup_metadata(monitored_name: str, df: pd.DataFrame) -> dict:
    prior = df[df['monitored'] == monitored_name]
    if prior.empty:
        return {c: '' for c in META_COLS}
    return {c: prior.sort_values('bid_date').iloc[-1].get(c, '') for c in META_COLS}


def _opportunity(monitored_name: str, df: pd.DataFrame, before_dt) -> str:
    prior = df[(df['monitored'] == monitored_name) & (df['bid_date'] < before_dt)]
    if prior.empty:
        return 'new'
    return pd.Timestamp(prior['bid_date'].max()).strftime('%-m/%-d/%Y')


def update_constraints(start_dt: str, end_dt: str, save: bool = True) -> pd.DataFrame:
    df = pd.read_csv(CSV_PATH)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={'constraints': 'monitored', 'constraints ': 'monitored'})
    df['monitored'] = df['monitored'].str.strip()
    df['bid_date']  = pd.to_datetime(df['bid_date'], format='mixed')

    date_range = pd.date_range(start=start_dt, end=end_dt, freq='D')
    new_rows   = []

    for dt in date_range:
        dt_str = dt.strftime('%Y-%m-%d')
        print(f"\nFetching {dt_str}...")

        fetched = _fetch_mvalues(dt_str)
        fetched = fetched[
            (fetched['DA_mvalue'] <= THRESHOLD) | (fetched['RT_mvalue'] <= THRESHOLD)
        ].reset_index(drop=True)

        if fetched.empty:
            print(f"  no constraints below threshold")
            continue
        print(f"  {len(fetched)} constraint(s) found")

        existing_mask = df['bid_date'] == dt

        if existing_mask.any():
            for _, frow in fetched.iterrows():
                row_mask = existing_mask & (df['monitored'] == frow['monitored'])
                opp = _opportunity(frow['monitored'], df, before_dt=dt)
                if row_mask.any():
                    df.loc[row_mask, 'DA_mvalue']   = frow['DA_mvalue']
                    df.loc[row_mask, 'RT_mvalue']   = frow['RT_mvalue']
                    df.loc[row_mask, 'opportunity'] = opp
                    print(f"    updated  : {frow['monitored']} (opportunity={opp})")
                else:
                    meta = _lookup_metadata(frow['monitored'], df)
                    new_rows.append({'bid_date': dt, 'monitored': frow['monitored'],
                                     'DA_mvalue': frow['DA_mvalue'], 'RT_mvalue': frow['RT_mvalue'],
                                     **meta, "today's wind": '', 'opportunity': opp})
                    print(f"    appended : {frow['monitored']} (new for this date, opportunity={opp})")
        else:
            for _, frow in fetched.iterrows():
                meta = _lookup_metadata(frow['monitored'], df)
                opp  = _opportunity(frow['monitored'], df, before_dt=dt)
                new_rows.append({'bid_date': dt, 'monitored': frow['monitored'],
                                 'DA_mvalue': frow['DA_mvalue'], 'RT_mvalue': frow['RT_mvalue'],
                                 **meta, "today's wind": '', 'opportunity': opp})
                print(f"    appended : {frow['monitored']} (opportunity={opp})")

    result = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    result['bid_date'] = pd.to_datetime(result['bid_date'], format='mixed')

    # Sort: by date first, then within each date by abs(RT - DA) descending
    result['_rank'] = (
        pd.to_numeric(result['RT_mvalue'], errors='coerce').fillna(0) -
        pd.to_numeric(result['DA_mvalue'], errors='coerce').fillna(0)
    ).abs()
    result = (result
              .sort_values(['bid_date', '_rank'], ascending=[True, False])
              .drop(columns='_rank')
              .reset_index(drop=True))

    result['bid_date'] = result['bid_date'].dt.strftime('%-m/%-d/%Y')
    result = result[COL_ORDER]

    print(f"\n{'='*60}")
    print(f"Total rows: {len(result)}  |  New rows added: {len(new_rows)}")
    display(result.tail(len(new_rows) + 3))

    if save:
        result = result.rename(columns={'monitored': 'constraints '})
        result.to_csv(CSV_PATH, index=False)
        print(f"Saved → {CSV_PATH}")

    return result

In [4]:
today    = pd.Timestamp.now(tz='US/Central').normalize().tz_localize(None)
start_dt = (today - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
end_dt   = (today + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

result = update_constraints(start_dt, end_dt, save=True)


Fetching 2026-06-08...
  18 constraint(s) found
    updated  : lncimarron-haymakr4 (opportunity=new)
    updated  : lnctnwd1-pine (opportunity=6/4/2026)
    updated  : lndovrt-turcrk2 (opportunity=new)
    updated  : lnlenexa-shwnmsn5 (opportunity=6/5/2026)
    updated  : lnmaud_tap-earlsbrh (opportunity=6/2/2026)
    appended : lnnebrcty-sub3456 (new for this date, opportunity=new)
    updated  : lnommarlo4-rsh_sprg_rev23 (opportunity=new)
    updated  : lnosage_og-webbtap4 (opportunity=6/5/2026)
    updated  : lnrussett-sbrown (opportunity=6/7/2026)
    appended : lnsshreve-wallace3 (new for this date, opportunity=new)
    updated  : lnstilwell-redel5 (opportunity=6/5/2026)
    appended : lnsugr_crk-sub_h (new for this date, opportunity=new)
    updated  : lnsw_sta-anadarko (opportunity=new)
    appended : lnwashit1-sw_sta (new for this date, opportunity=new)
    updated  : lnweav-tallgras (opportunity=6/5/2026)
    updated  : xfmrduncan-duncan (opportunity=6/7/2026)
    updated  : 

,bid_date,monitored,DA_mvalue,RT_mvalue,location,physical_condition,outage_name,start_date,end_date,wind,reserve_zone,today's wind,opportunity
144,6/9/2026,lngord-maiz,-1666.0,-313.0,"Wichita, Kansas",high wind plus small outage,"Walters City (OMPA) - Walters Junction 69 kV, ...",2026-05-26 7:32,2026-06-05 15:00,high,4,NaN,6/4/2026
145,6/9/2026,lnommarlo4-rsh_sprg_rev23,-1312.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6/8/2026
146,6/9/2026,lntulsa_no-46_st_tp,0.0,-1309.0,,,,,,,,,new
147,6/9/2026,lnpenn1-sntfe,-1216.0,-38.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,new
148,6/9/2026,lnsshreve-wallace3,0.0,-1156.0,,,,,,,,,new
149,6/9/2026,lnjay_hawk-frankln5,-1138.0,-47.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,new
150,6/9/2026,lnfirstcrk-roanrdge,-4395.0,-5477.0,Near Kansa City,"RT binds more in 13-14, 22-24, DA binds more o...","Hallmark MPS - Liberty West 161 kV, 5/19-6-16,...",5/19,6/16,high,4,NaN,6/7/2026
151,6/9/2026,lnwardwa-bismark2_rev23,-916.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,new
152,6/9/2026,lnwashit1-sw_sta,-1505.0,-933.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,new
153,6/9/2026,xfmrduncan-duncan,-5360.0,-4831.0,"South OKGE, Lawton","high load driven, ng generation up, 64kv","Walters City (OMPA) - Walters Junction 69 kV, +7%",2026-06-10 8:00,2026-06-10 16:00,NaN,2,NaN,6/8/2026


Saved → /var/www/python/Qingcheng/QCTest/Manual_bidding/update_sheet/Daily Bidding - daily_constraint.csv


In [3]:
result

,bid_date,constraints,DA_mvalue,RT_mvalue,location,physical_condition,outage_name,start_date,end_date,wind,reserve_zone,today's wind,opportunity
0,5/28/2026,lnjarb-166thstr,0.0,-6898.0,KCPL,high wind,"Midland Junction - Pentagon 115 kV, 6/19/2026",3/9,6/19,high,NaN,NaN,NaN
1,5/28/2026,lnpayne_wf-paoli2,-3800.0,-7500.0,South OKGE,high wind,"Johnston County - Sunnyside 345kV, 6/2/2026, +...",1/9,6/5,high,"3,4",NaN,NaN
2,5/29/2026,lnfirstcrk-roanrdge,-1000.0,-7000.0,Near Kansa City,"RT binds more in 13-14, 22-24, DA binds more o...","Hallmark MPS - Liberty West 161 kV, 5/19-6-16,...",5/19,6/16,high,4,"1: ow: 44, fw: 21, ol: 92, fl: 92\n2: ow: 45, ...",NaN
3,5/29/2026,lnhuron-hurontap,-200.0,-2700.0,East of South Dakota,DA binds more during hour 11-17 and RT binds m...,"Hay Creek - North Bismark 115 kv, 5/26-6/5, +2...",5/26,6/5,NaN,5,NaN,NaN
4,5/29/2026,lncorntp4-naples1,-1331.0,-3600.0,South OKGE,high wind driven,"Johnston County - Sunnyside 345kV, 6/2/2026, +...",1/9,6/5,high,"3,4",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142,6/9/2026,lnjay_hawk-frankln5,-1138.0,0.0,,,,,,,,,new
143,6/9/2026,lndawsonc-lewiswp,-1118.0,0.0,,,,,,,,,new
144,6/9/2026,xfmrsiouxcy-siouxcy,-1116.0,0.0,Sioux City Nebraska,high wind driven,NaN,NaN,NaN,high,1,,6/6/2026
145,6/9/2026,lnwardwa-bismark2_rev23,-916.0,0.0,,,,,,,,,new
